# NER and Entity Resolution - Baseline spaCy
This work is a part of the knowledge graph pipeline built for 26 Spring BigCo Studio at Cornell Tech, and belongs to Team 10.

The whole process of this part is run on Google Colab.

# 1. Set up GDrive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
%cd /content/gdrive/MyDrive/NER_EntityResolution

/content/gdrive/MyDrive/NER_EntityResolution


# 2. Install required packages

In [ ]:
!pip install -q pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 3. Preprocess data

In [ ]:
!python src/preprocess.py \
  --input data/mock/mock2_input.csv \
  --cleaned-output data/processed/cleaned_input_mock2.csv \
  --invalid-output data/processed/invalid_rows_mock2.csv

[preprocess] Wrote 53 valid row(s) to: data/processed/cleaned_input_mock2.csv
[preprocess] Wrote 7 invalid row(s) to: data/processed/invalid_rows_mock2.csv


In [ ]:
!ls data/processed/

cleaned_input_mock1.csv  invalid_rows_mock1.csv
cleaned_input_mock2.csv  invalid_rows_mock2.csv


In [ ]:
import pandas as pd

cleaned_df = pd.read_csv("data/processed/cleaned_input_mock2.csv")
invalid_df = pd.read_csv("data/processed/invalid_rows_mock2.csv")

print("Cleaned input:")
display(cleaned_df)

print("Invalid rows:")
display(invalid_df)

Cleaned input:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,1,SEC,Pfizer acquired Seagen in 2023.,https://example.com/sec1,2023-01,Pfizer,SEC,Pfizer acquired Seagen in 2023.,https://example.com/sec1,2023-01,Pfizer,False,NaN,NaN,True
1,2,SEC,Merck entered into a license agreement with Mo...,https://example.com/sec2,2024-02,Merck,SEC,Merck entered into a license agreement with Mo...,https://example.com/sec2,2024-02,Merck,False,NaN,NaN,True
2,3,USPTO,Regeneron and Sanofi jointly developed an anti...,https://example.com/uspto1,2022-06,Regeneron,USPTO,Regeneron and Sanofi jointly developed an anti...,https://example.com/uspto1,2022-06,Regeneron,False,NaN,NaN,True
3,4,PUBMED,CSL Behring collaborated with Takeda on plasma...,https://example.com/pubmed1,2021-11,CSL Behring,PUBMED,CSL Behring collaborated with Takeda on plasma...,https://example.com/pubmed1,2021-11,CSL Behring,False,NaN,NaN,True
4,5,SEC,Bristol Myers Squibb purchased Karuna Therapeu...,https://example.com/sec3,2024-12,Bristol Myers Squibb,SEC,Bristol Myers Squibb purchased Karuna Therapeu...,https://example.com/sec3,2024-12,Bristol Myers Squibb,False,NaN,NaN,True
5,6,USPTO,AbbVie filed patent with Genmab for a bispecif...,https://example.com/uspto2,2020-09,AbbVie,USPTO,AbbVie filed patent with Genmab for a bispecif...,https://example.com/uspto2,2020-09,AbbVie,False,NaN,NaN,True
6,7,SEC,AstraZeneca partnered with Daiichi Sankyo to c...,https://example.com/sec4,2023-08,AstraZeneca,SEC,AstraZeneca partnered with Daiichi Sankyo to c...,https://example.com/sec4,2023-08,AstraZeneca,False,NaN,NaN,True
7,8,PUBMED,Novartis worked with Alnylam on RNA interferen...,https://example.com/pubmed2,2022-03,Novartis,PUBMED,Novartis worked with Alnylam on RNA interferen...,https://example.com/pubmed2,2022-03,Novartis,False,NaN,NaN,True
8,9,SEC,10X GENOMICS INC acquired a small diagnostics ...,https://example.com/sec5,2024-03,10X Genomics,SEC,10X GENOMICS INC acquired a small diagnostics ...,https://example.com/sec5,2024-03,10X Genomics,False,NaN,NaN,True
9,10,SEC,10X GENOMICS BV entered into a collaboration w...,https://example.com/sec6,2024-04,10X Genomics,SEC,10X GENOMICS BV entered into a collaboration w...,https://example.com/sec6,2024-04,10X Genomics,False,NaN,NaN,True


Invalid rows:


,source_id,source_type,raw_text,source_url,date,company_seed,source_type_cleaned,raw_text_cleaned,source_url_cleaned,date_cleaned,company_seed_cleaned,company_seed_invalid,company_seed_invalid_reason,row_invalid_reasons,row_valid_for_ner
0,23,SEC,#NAME? filed a report.,https://example.com/bad1,2021-07,#NAME?,SEC,#NAME? filed a report.,https://example.com/bad1,2021-07,#NAME?,True,excel_error_token,excel_error_token,False
1,24,USPTO,11-JUL appears in the assignee field.,https://example.com/bad2,2020-05,11-JUL,USPTO,11-JUL appears in the assignee field.,https://example.com/bad2,2020-05,11-JUL,True,date_like_token,date_like_token,False
2,25,SEC,NaN,https://example.com/bad3,2022-02,Unknown,SEC,NaN,https://example.com/bad3,2022-02,Unknown,False,NaN,missing_raw_text,False
3,26,PUBMED,,https://example.com/bad4,2023-01,BlankCo,PUBMED,NaN,https://example.com/bad4,2023-01,BlankCo,False,NaN,missing_raw_text,False
4,27,SEC,N/A entered into a collaboration.,https://example.com/bad5,2022-06,NaN,SEC,N/A entered into a collaboration.,https://example.com/bad5,2022-06,NaN,True,missing_company_seed,missing_company_seed,False
5,28,USPTO,NULL filed patent with ???.,https://example.com/bad6,2021-09,NaN,USPTO,NULL filed patent with ???.,https://example.com/bad6,2021-09,NaN,True,missing_company_seed,missing_company_seed,False
6,50,UNKNOWN,Omega Therapeutics acquired ExampleCo.,https://example.com/unknown1,2022-07,Omega Therapeutics,UNKNOWN,Omega Therapeutics acquired ExampleCo.,https://example.com/unknown1,2022-07,Omega Therapeutics,False,NaN,unknown_source_type,False


# 4. Run spaCy baseline for NER

In [ ]:
!python src/ner_baseline.py \
  --input data/processed/cleaned_input_mock2.csv \
  --output outputs/ner/ner_results_mock2_baseline.csv \
  --model en_core_web_sm

[ner] Wrote 58 entity mention(s) to: outputs/ner/ner_results_mock2_baseline.csv


In [ ]:
ner_df = pd.read_csv("outputs/ner/ner_results_mock2_baseline.csv")
display(ner_df)

,source_id,source_type,source_type_cleaned,source_url,source_url_cleaned,date,date_cleaned,company_seed,company_seed_cleaned,raw_text,raw_text_cleaned,raw_mention,entity_label,start_char,end_char
0,1,SEC,SEC,https://example.com/sec1,https://example.com/sec1,2023-01,2023-01,Pfizer,Pfizer,Pfizer acquired Seagen in 2023.,Pfizer acquired Seagen in 2023.,Seagen,ORG,16,22
1,3,USPTO,USPTO,https://example.com/uspto1,https://example.com/uspto1,2022-06,2022-06,Regeneron,Regeneron,Regeneron and Sanofi jointly developed an anti...,Regeneron and Sanofi jointly developed an anti...,Sanofi,ORG,14,20
2,4,PUBMED,PUBMED,https://example.com/pubmed1,https://example.com/pubmed1,2021-11,2021-11,CSL Behring,CSL Behring,CSL Behring collaborated with Takeda on plasma...,CSL Behring collaborated with Takeda on plasma...,CSL Behring,ORG,0,11
3,5,SEC,SEC,https://example.com/sec3,https://example.com/sec3,2024-12,2024-12,Bristol Myers Squibb,Bristol Myers Squibb,Bristol Myers Squibb purchased Karuna Therapeu...,Bristol Myers Squibb purchased Karuna Therapeu...,Bristol Myers Squibb,ORG,0,20
4,5,SEC,SEC,https://example.com/sec3,https://example.com/sec3,2024-12,2024-12,Bristol Myers Squibb,Bristol Myers Squibb,Bristol Myers Squibb purchased Karuna Therapeu...,Bristol Myers Squibb purchased Karuna Therapeu...,Karuna Therapeutics,ORG,31,50
5,7,SEC,SEC,https://example.com/sec4,https://example.com/sec4,2023-08,2023-08,AstraZeneca,AstraZeneca,AstraZeneca partnered with Daiichi Sankyo to c...,AstraZeneca partnered with Daiichi Sankyo to c...,AstraZeneca,ORG,0,11
6,7,SEC,SEC,https://example.com/sec4,https://example.com/sec4,2023-08,2023-08,AstraZeneca,AstraZeneca,AstraZeneca partnered with Daiichi Sankyo to c...,AstraZeneca partnered with Daiichi Sankyo to c...,Daiichi Sankyo,ORG,27,41
7,8,PUBMED,PUBMED,https://example.com/pubmed2,https://example.com/pubmed2,2022-03,2022-03,Novartis,Novartis,Novartis worked with Alnylam on RNA interferen...,Novartis worked with Alnylam on RNA interferen...,Novartis,ORG,0,8
8,8,PUBMED,PUBMED,https://example.com/pubmed2,https://example.com/pubmed2,2022-03,2022-03,Novartis,Novartis,Novartis worked with Alnylam on RNA interferen...,Novartis worked with Alnylam on RNA interferen...,RNA,ORG,32,35
9,9,SEC,SEC,https://example.com/sec5,https://example.com/sec5,2024-03,2024-03,10X Genomics,10X Genomics,10X GENOMICS INC acquired a small diagnostics ...,10X GENOMICS INC acquired a small diagnostics ...,GENOMICS INC,ORG,4,16


# 5. Run alias resolution

In [ ]:
!python src/resolve_alias.py \
  --input outputs/ner/ner_results_mock2_baseline.csv \
  --resolved-output outputs/entity_resolution/resolved_entities_mock2_baseline.csv \
  --review-output outputs/entity_resolution/review_queue_mock2_baseline.csv

[resolve_alias] Wrote 58 resolved row(s) to: outputs/entity_resolution/resolved_entities_mock2_baseline.csv
[resolve_alias] Wrote 3 review pair(s) to: outputs/entity_resolution/review_queue_mock2_baseline.csv


In [ ]:
import os

for path in [
    "outputs/entity_resolution/resolved_entities_mock2_baseline.csv",
    "outputs/entity_resolution/review_queue_mock2_baseline.csv",
]:
    print(path)
    print("exists:", os.path.exists(path))
    if os.path.exists(path):
        print("size:", os.path.getsize(path), "bytes")
    print("-" * 40)

outputs/entity_resolution/resolved_entities_mock2_baseline.csv
exists: True
size: 13889 bytes
----------------------------------------
outputs/entity_resolution/review_queue_mock2_baseline.csv
exists: True
size: 421 bytes
----------------------------------------


# 6. Candidate relations

In [ ]:
import pandas as pd

resolved_df = pd.read_csv("outputs/entity_resolution/resolved_entities_mock2_baseline.csv")
display(resolved_df)

,source_id,source_type,source_url,date,company_seed,raw_text,raw_mention,entity_label,start_char,end_char,normalized_name,base_name,legal_suffixes,canonical_group_id,canonical_name,merge_confidence,merge_decision,evidence_note
0,1,SEC,https://example.com/sec1,2023-01,Pfizer,Pfizer acquired Seagen in 2023.,Seagen,ORG,16,22,SEAGEN,SEAGEN,NaN,42,Seagen,1.00,singleton,singleton_no_merge_needed
1,3,USPTO,https://example.com/uspto1,2022-06,Regeneron,Regeneron and Sanofi jointly developed an anti...,Sanofi,ORG,14,20,SANOFI,SANOFI,NaN,21,Sanofi,1.00,singleton,singleton_no_merge_needed
2,4,PUBMED,https://example.com/pubmed1,2021-11,CSL Behring,CSL Behring collaborated with Takeda on plasma...,CSL Behring,ORG,0,11,CSL BEHRING,CSL BEHRING,NaN,11,CSL Behring,0.88,auto_merge,auto_merge_component
3,5,SEC,https://example.com/sec3,2024-12,Bristol Myers Squibb,Bristol Myers Squibb purchased Karuna Therapeu...,Bristol Myers Squibb,ORG,0,20,BRISTOL MYERS SQUIBB,BRISTOL MYERS SQUIBB,NaN,9,Bristol-Myers Squibb,1.00,singleton,singleton_no_merge_needed
4,5,SEC,https://example.com/sec3,2024-12,Bristol Myers Squibb,Bristol Myers Squibb purchased Karuna Therapeu...,Karuna Therapeutics,ORG,31,50,KARUNA THERAPEUTICS,KARUNA THERAPEUTICS,NaN,40,Karuna Therapeutics,1.00,singleton,singleton_no_merge_needed
5,7,SEC,https://example.com/sec4,2023-08,AstraZeneca,AstraZeneca partnered with Daiichi Sankyo to c...,AstraZeneca,ORG,0,11,ASTRAZENECA,ASTRAZENECA,NaN,37,AstraZeneca,1.00,singleton,singleton_no_merge_needed
6,7,SEC,https://example.com/sec4,2023-08,AstraZeneca,AstraZeneca partnered with Daiichi Sankyo to c...,Daiichi Sankyo,ORG,27,41,DAIICHI SANKYO,DAIICHI SANKYO,NaN,31,Daiichi Sankyo,1.00,singleton,singleton_no_merge_needed
7,8,PUBMED,https://example.com/pubmed2,2022-03,Novartis,Novartis worked with Alnylam on RNA interferen...,Novartis,ORG,0,8,NOVARTIS,NOVARTIS,NaN,39,Novartis,1.00,singleton,singleton_no_merge_needed
8,8,PUBMED,https://example.com/pubmed2,2022-03,Novartis,Novartis worked with Alnylam on RNA interferen...,RNA,ORG,32,35,RNA,RNA,NaN,13,RNA,1.00,singleton,singleton_no_merge_needed
9,9,SEC,https://example.com/sec5,2024-03,10X Genomics,10X GENOMICS INC acquired a small diagnostics ...,GENOMICS INC,ORG,4,16,GENOMICS INC,GENOMICS,INC,44,GENOMICS INC,1.00,singleton,singleton_no_merge_needed


In [ ]:
!mkdir -p outputs/relations

!python src/extract_candidate_relations.py \
  --input outputs/entity_resolution/resolved_entities_mock2_baseline.csv \
  --output outputs/relations/candidate_relations_mock2_baseline.csv

[extract_candidate_relations] Wrote 13 candidate row(s) to: outputs/relations/candidate_relations_mock2_baseline.csv
